# Case 3

# How does the prediction threshold affect detection over time?

## Purpose of this notebook

This notebook demonstrates an important streaming anomaly-detection problem explored in the thesis:

> A prediction threshold that works well overall may still behave poorly at different stages of the stream.

The thesis studies anomaly detection in streaming ERA5 weather data using several VAE architectures. After a model produces an anomaly score, that score is converted into a prediction using a decision threshold. The choice of threshold affects the balance between **false alarms** and **missed anomalies**.

This testcase focuses on **group anomalies** in window mode and examines how different prediction thresholds behave as the stream progresses.

---

## What is being compared?

The notebook compares several threshold settings for two sequential VAE architectures:

| Variant | Prediction threshold | What it illustrates |
|---|---:|---|
| **Transformer-VAE** | 0.3 | A low threshold that may flag too many observations early in the stream. |
| **Transformer-VAE** | 0.6 | A more balanced reference setting. |
| **Transformer-VAE** | 0.9 | A high threshold that requires stronger evidence before predicting an anomaly. |
| **LSTM-VAE** | 0.9 | A high-threshold example that can miss anomalies later in the stream. |

The important point is not simply which threshold gives the highest overall F1. This testcase shows that **threshold behaviour can change over time as the model and adaptive score distribution evolve**.

## Thesis result being illustrated

The thesis observed two opposite failure modes:

- With a **low prediction threshold**, Transformer-VAE may over-predict anomalies early in the stream, producing false positives before its score calibration stabilises.
- With a **high prediction threshold**, LSTM-VAE may remain precise but eventually lose recall when anomaly confidence falls below the decision boundary.

The result illustrates why a single aggregate metric can hide important streaming behaviour. Thresholds should therefore be inspected **across the stream**, not only through final F1.

This reduced notebook uses a shorter ERA5 period than the full thesis experiments. Exact metric values are not expected to match the thesis. The important result is the **qualitative behaviour of the threshold settings over time**.

---

## What this notebook will do

1. Check whether a GPU is available.
2. Clone and install the thesis repository.
3. Run the threshold-calibration variants on the same reduced ERA5 stream.
4. Build a common performance comparison.
5. Inspect F1, precision, recall, and confusion-matrix counts.
6. Plot F1 across equal sections of the stream to expose changes over time.

### Expected runtime


The configured runs use window-based streaming and can use a GPU when available. Runtime depends on the Colab GPU assigned to the session and may differ between users.


### Before running


This notebook reads code and data from a **private GitHub repository**. You must:

1. Have permission to access the repository.
2. Create a fine-grained GitHub token with read-only access to this repository.
3. In Colab, open the **Secrets** panel using the key icon.
4. Add the secret `GITHUB_TOKEN`.
5. Enable **Notebook access** for that secret.
6. Select **Runtime → Change runtime type → GPU**.
7. Choose **Runtime → Run all**.

The source ERA5 data and its licence information are described in `DATA_LICENSE.md`.

# Run Case

### Check GPU availability

In [1]:
import torch

print("Environment check")
print("-----------------")
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Selected device:", torch.cuda.get_device_name(0))
else:
    print(
        "No GPU was detected. The notebook can still run, but "
        "window-based training may be slower."
    )


Environment check
-----------------
PyTorch version: 2.10.0+cu126
CUDA available: True
Selected device: NVIDIA RTX A5000


### Import and/or load Repo

In [2]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/hadasecohen/streaming-vae-anomaly-detection.git"
REPO_DIR = Path("/content/repo") if "COLAB_RELEASE_TAG" in os.environ else Path("repo")

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import userdata

    token = userdata.get("GITHUB_TOKEN")
    if not token:
        raise RuntimeError(
            "The Colab secret GITHUB_TOKEN is unavailable. "
            "Open the key icon in the left sidebar, add the token, "
            "and enable Notebook access."
        )

    if not REPO_DIR.exists():
        # Use an askpass helper so the token is not stored in the Git remote URL.
        askpass = Path("/content/git_askpass.sh")
        askpass.write_text(
            '#!/bin/sh\n'
            'case "$1" in\n'
            '  *Username*) echo "x-access-token" ;;\n'
            '  *Password*) echo "$GITHUB_TOKEN" ;;\n'
            'esac\n'
        )
        askpass.chmod(0o700)

        env = os.environ.copy()
        env["GITHUB_TOKEN"] = token
        env["GIT_ASKPASS"] = str(askpass)
        env["GIT_TERMINAL_PROMPT"] = "0"

        try:
            subprocess.run(
                ["git", "clone", REPO_URL, str(REPO_DIR)],
                check=True,
                env=env,
            )
        finally:
            askpass.unlink(missing_ok=True)

    os.chdir(REPO_DIR)
else:
    # When launched from notebooks/cases inside a local checkout,
    # move to the repository root.
    if not Path("run_regression.py").exists():
        os.chdir("../..")

print("Working directory:", os.getcwd())
print("Installing the project dependencies...")
subprocess.run(
    ["python", "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True,
)
%env MPLBACKEND=Agg
print("Setup complete.")

Working directory: /home/cohenhada/streaming-vae-anomaly-detection
Installing the project dependencies...
env: MPLBACKEND=Agg
Setup complete.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


### Step 1 — Run the experiment variants

#### Configuration file

The suite configuration is stored in:

```text
notebooks/cases/case03_threshold_calibration_suite.yaml
```

It combines the shared ERA5 configuration with the architecture, group-anomaly, window-stream, and prediction-threshold settings for each run.

All variants are evaluated on the same reduced dataset. The intended comparison is therefore the effect of the **prediction threshold** and how that effect changes during the stream.

#### Output Directory

During execution, the script creates a separate run directory for each variant under:

```text
runs/regression/case03_threshold_calibration/
```

You may see detailed training output below. The consolidated comparison is presented after all runs finish.

`notebooks/cases/case03_threshold_calibration_suite.yaml` layers `modules/era5_common.yaml` with the architecture / anomaly-type / stream-mode fragments for each of the runs below (see the suite file for the exact overrides). Runs execute sequentially. Window-mode runs are batched and fast (well under a minute each on GPU). Point-mode runs stream one gradient step per row with no batching, so they run on CPU instead (faster than GPU for this access pattern) and take a few minutes each — `modules/stream/point.yaml` caps them to a 5,000-row subset for this reason.

#### Expected output

The command below trains and evaluates each configured variant. Successful completion should produce two run folders containing predictions, metrics, and diagnostic outputs.

### Command:

In [3]:
!python run_regression.py notebooks/cases/case03_threshold_calibration_suite.yaml \
    --session runs/regression/case03_threshold_calibration

[suite] log     : /home/cohenhada/streaming-vae-anomaly-detection/runs/regression/case03_threshold_calibration/suite.log
[suite] suite   : /home/cohenhada/streaming-vae-anomaly-detection/notebooks/cases/case03_threshold_calibration_suite.yaml
[suite] session : /home/cohenhada/streaming-vae-anomaly-detection/runs/regression/case03_threshold_calibration
[suite] base cfg: /home/cohenhada/streaming-vae-anomaly-detection/notebooks/cases/../../modules/era5_common.yaml
[suite] module  : regression.run_trial
[suite] GPUs    : [0, 1]  (2 slot(s))
[suite] 4 run(s) planned:
  [  1] TF_low_threshold  [case03_threshold_calibration/case03_threshold_calibration_suite]  (module: regression.run_trial)
  [  2] TF_baseline_threshold  [case03_threshold_calibration/case03_threshold_calibration_suite]  (module: regression.run_trial)
  [  3] TF_high_threshold  [case03_threshold_calibration/case03_threshold_calibration_suite]  (module: regression.run_trial)
  [  4] LSTM_high_threshold  [case03_threshold_calib

### Step 2 — Build a common comparison

#### The next command reads the predictions from every run and produces:

- a common performance table;
- anomaly-score histograms;
- an F1 comparison across variants;
- confusion-matrix summaries; and
- seed-stability diagnostics, where applicable.

These outputs are saved under:

```text
runs/regression/case03_threshold_calibration/cross_compare/
```


### Command:

In [4]:
!python cross_compare.py runs/regression/case03_threshold_calibration

[cross] session : runs/regression/case03_threshold_calibration
[cross] output  : runs/regression/case03_threshold_calibration/cross_compare
[cross] 4 run(s) found:
  LSTM_Group_0.9                                      arch=LSTM  anomaly=Group  cyclic=False  mode=window
  TF_VAE_Group_0.3                                    arch=TF_VAE  anomaly=Group  cyclic=False  mode=window
  TF_VAE_Group_0.6                                    arch=TF_VAE  anomaly=Group  cyclic=False  mode=window
  TF_VAE_Group_0.9                                    arch=TF_VAE  anomaly=Group  cyclic=False  mode=window

[cross] Loading run data ...
  LSTM_Group_0.9 ... ok
  TF_VAE_Group_0.3 ... ok
  TF_VAE_Group_0.6 ... ok
  TF_VAE_Group_0.9 ... ok

[cross] Writing outputs ...
  saved performance_table.csv
  saved performance_summary.txt

  ══ Group anomalies ═══════════════════════════════════════════════════════════════════════════════════════════════
  run_name                                     arch       variant

### Step 3 — Inspect the numerical results

#### Focus first on `f1`, `precision`, and `recall`.

- **F1** summarises the balance between precision and recall.
- **Precision** decreases when many normal observations are incorrectly flagged as anomalies.
- **Recall** decreases when true anomalies are missed.
- `tp`, `fp`, and `fn` help identify which type of error is responsible.

#### Expected qualitative pattern

The aggregate table is useful, but it does **not** tell the whole story in this testcase.

A low threshold may suffer from excess false positives, while a high threshold may preserve precision at the cost of recall. More importantly, these effects may occur at different parts of the stream.

The next step therefore examines performance **over time**, rather than relying only on the final aggregate metrics.


In [5]:
import pandas as pd
from IPython.display import display

performance_path = (
    "runs/regression/case03_threshold_calibration/"
    "cross_compare/performance_table.csv"
)
perf = pd.read_csv(performance_path)

columns = [
    "run_name", "arch", "anomaly", "variant",
    "f1", "auc", "precision", "recall",
    "tp", "fp", "fn",
]

print("Comparison of prediction-threshold variants")
display(
    perf[columns]
    .sort_values("f1", ascending=False)
    .reset_index(drop=True)
    .style.format({
        "f1": "{:.3f}",
        "auc": "{:.3f}",
        "precision": "{:.3f}",
        "recall": "{:.3f}",
    })
)


Comparison of prediction-threshold variants


,run_name,arch,anomaly,variant,f1,auc,precision,recall,tp,fp,fn
0,TF_VAE_Group_0.3,TF_VAE,Group,0.300000,0.359,0.752,0.221,0.972,860,3039,25
1,TF_VAE_Group_0.6,TF_VAE,Group,0.600000,0.323,0.808,0.779,0.203,180,51,705
2,LSTM_Group_0.9,LSTM,Group,0.900000,0.011,0.830,1.000,0.006,5,0,880
3,TF_VAE_Group_0.9,TF_VAE,Group,0.900000,0.009,0.812,1.000,0.004,4,0,881


### Step 4 — Interpret performance across the stream

The aggregate metrics above combine the entire test stream into one number. This can hide **when** a threshold fails.

The plot below divides each run into eight equal-length stream segments and computes F1 separately within each segment.

When reading the plot, ask:

1. Does the low Transformer-VAE threshold perform poorly near the beginning and then recover?
2. Does the higher Transformer-VAE threshold reduce early instability?
3. Does the LSTM-VAE with a high threshold lose F1 later in the stream?
4. If F1 falls, is the likely cause excessive false positives or missed anomalies?

### Expected qualitative pattern

The thesis observed two different calibration problems:

```text
Low threshold  →  early over-detection / false positives
High threshold →  conservative predictions / possible recall loss
```

The purpose of this plot is to make those **temporal failure modes** visible.

## Main takeaway

Prediction-threshold selection is not only a static precision–recall trade-off. In an online VAE, model updates and adaptive score calibration change the distribution seen during the stream. A threshold that appears reasonable from an overall metric can therefore be poorly calibrated during particular periods.

This is why the thesis evaluates threshold behaviour over time in addition to reporting aggregate F1.


In [6]:
# Custom view: F1 over equal-length stream segments.
# This approximates the thesis's anomaly-equal bucket plots by dividing
# each reduced run into equal-length sections and calculating F1 per section.

import glob
import pandas as pd
import matplotlib.pyplot as plt

SESSION = "runs/regression/case03_threshold_calibration"
N_BUCKETS = 8

run_globs = {
    "Transformer threshold=0.3": (
        f"{SESSION}/case03_threshold_calibration/"
        "case03_threshold_calibration_suite/Group/TF_VAE/"
        "threshold_0_3/*_seed_42/trial_predictions.csv"
    ),
    "Transformer threshold=0.6": (
        f"{SESSION}/case03_threshold_calibration/"
        "case03_threshold_calibration_suite/Group/TF_VAE/"
        "threshold_0_6/*_seed_42/trial_predictions.csv"
    ),
    "Transformer threshold=0.9": (
        f"{SESSION}/case03_threshold_calibration/"
        "case03_threshold_calibration_suite/Group/TF_VAE/"
        "threshold_0_9/*_seed_42/trial_predictions.csv"
    ),
    "LSTM threshold=0.9": (
        f"{SESSION}/case03_threshold_calibration/"
        "case03_threshold_calibration_suite/Group/LSTM/"
        "threshold_0_9/*_seed_42/trial_predictions.csv"
    ),
}

runs = {}
for label, pattern in run_globs.items():
    matches = sorted(glob.glob(pattern))
    if not matches:
        raise FileNotFoundError(
            f"No prediction file found for {label}. Expected pattern:\n{pattern}"
        )
    runs[label] = matches[0]


def bucket_f1(csv_path, n_buckets):
    df = pd.read_csv(
        csv_path,
        skiprows=1,
        header=None,
        names=[
            "ts_start", "ts_end", "step", "confusion", "label",
            "final_pred", "trained", "final_conf", "metrics"
        ],
    )

    df["bucket"] = pd.cut(df["step"], n_buckets, labels=False)
    output = [float("nan")] * n_buckets

    for bucket, group in df.groupby("bucket"):
        tp = (group["confusion"] == "TP").sum()
        fp = (group["confusion"] == "FP").sum()
        fn = (group["confusion"] == "FN").sum()

        precision = tp / (tp + fp) if (tp + fp) else float("nan")
        recall = tp / (tp + fn) if (tp + fn) else float("nan")

        if (
            precision == precision
            and recall == recall
            and (precision + recall) > 0
        ):
            f1 = 2 * precision * recall / (precision + recall)
        else:
            f1 = float("nan")

        output[int(bucket)] = f1

    return output


fig, ax = plt.subplots(figsize=(9, 4))

for label, path in runs.items():
    values = bucket_f1(path, N_BUCKETS)
    ax.plot(
        range(1, N_BUCKETS + 1),
        values,
        marker="o",
        label=label,
    )

ax.set_xlabel("Stream segment (1–8, equal length)")
ax.set_ylabel("F1 within segment")
ax.set_title("Prediction-threshold behaviour across the stream")
ax.set_ylim(-0.05, 1.05)
ax.legend()
plt.show()


---

## Scope of this testcase

This notebook is a compact demonstration, not a full reproduction of every threshold experiment reported in the thesis. It uses a shorter ERA5 interval and reduced-run settings so that readers can execute it on Colab.

The eight stream sections used above are an explanatory approximation of the thesis's bucket-based temporal analysis. Exact metric values and the exact position of performance changes may therefore differ from the full experiments.

Conclusions should be based on the **direction and timing of the threshold effects**, rather than exact numerical agreement with the thesis figures.

For the complete experiment definitions and full-scale results, consult the corresponding thesis threshold-calibration section together with the YAML configuration files used by this suite.
